# Prep work

In [1]:
import os
import pandas as pdß
import numpy as np
import pprint
import json
import cv2
import random

import pickle # Load refs and annotations
from typing import Any, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.utils.tensorboard import SummaryWriter

import torchvision
import torchmetrics


import pytorch_lightning as pl
from pytorch_lightning.utilities.types import STEP_OUTPUT

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import CLIPProcessor, CLIPModel

from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import clip
from ultralytics import YOLO
from PIL import Image, ImageDraw


/Users/mattiacarolo/.pyenv/versions/3.9.6/envs/bagigio/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.backends.mps.is_available():
    print("MPS backend is available.")
else:
    print("MPS backend is not available.")

MPS backend is available.


In [3]:
device = torch.device('mps')

In [4]:
### modelUtils.py ###

get_device_first_call=True
def get_device():
    global get_device_first_call
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if get_device_first_call:
        #info("The current device is " + device)
        get_device_first_call=False
    return device

def save_model(model, epoch, optimizer, total_loss, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': total_loss,
        }, path+"/personal_model_"+str(epoch)+".pt")

def load_model(model, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    return model, epoch, loss

class TensorBoard():
    # This class allows to log in tensorboard
    def __init__(self, log_dir):
        rootdir = log_dir
        max=1
        for file in os.listdir(rootdir):
            d = os.path.join(rootdir, file)
            if os.path.isdir(d) and file.startswith("exp"):
                num = int(file.replace("exp",""))
                if num > max:
                    max = num
        log_dir = log_dir+"/exp"+str(max+1)
        self.writer = SummaryWriter(log_dir=log_dir)

    def log_values(self, step, loss, accuracy, prefix):
        self.writer.add_scalar(f"{prefix}/loss", loss, step)
        self.writer.add_scalar(f"{prefix}/accuracy", accuracy, step)

    def close(self):
        self.writer.close()

# Dataset

In [5]:
class DataAugmentation():
    # This class is used to perform tranformation in order to have augmented data

    def blur(img):
        # Gaussian Blur the image
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.GaussianBlur(kernel_size=5),
            transforms.ToTensor(),
        ])
        img = transform(img)
        return img

    def rotate(img):
        # Rotate the image
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomRotation(degrees=180),
            transforms.ToTensor(),
        ])
        img = transform(img)
        return img
    def grayscale(img):
        # Convert the image to grayscale
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
        ])
        img = transform(img)
        return img
    def colorrand(img):
        # Randomly change the color of the image
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.ColorJitter(brightness=0.01*random.randrange(1,50), contrast=0.01*random.randrange(1,50), saturation=0.01*random.randrange(1,50), hue=0.01*random.randrange(1,50)),
            transforms.ToTensor(),
        ])
        img = transform(img)
        return img
    def random_crop(img):
        # Randomly crop the image
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomCrop((224,224)),
            transforms.ToTensor(),
        ])
        img = transform(img)
        return img

    def random_augmentation(img):
        # Randomly choose a transformation to be applied to the image
        n = random.randint(0, 5)
        if n == 0:
            return DataAugmentation.blur(img)
        elif n == 1:
            return DataAugmentation.rotate(img)
        elif n == 2:
            return DataAugmentation.grayscale(img)
        elif n == 3:
            return DataAugmentation.colorrand(img)
        else:
            return DataAugmentation.random_crop(img)

In [6]:
def getcaption(elem):
    li = []
    for e in elem["sentences"]:
        li.append(e['raw'])
    return li



class RefCOCOG(Dataset):
    """
    Args:
        The dataset will be the raw data wothput any tipe of preprocessing
        {
            'file_name':
            'caption':
            'ann_id': needed to extract the relative bbox from the .json file
            'bbox': values are set like following:
                - x
                - y
                - width
                - height
        }
    """
    def __init__(self, refs, annotations, split="train"):

        dataset = list()

        for elem in [d for d in refs if d["split"]==split]:
            file_name = os.path.join("./refcocog/images/", f'{"_".join(elem["file_name"].split("_")[:3])}.jpg')
            len_sent = len(elem['sentences'])
            sentences = elem['sentences']
            raws = []
            for i in sentences:
                raws.append(i['raw'])
            bbox = annotations[elem['ann_id']]
            for i in raws:
                dataset.append({"file_name":file_name,"raw":i,"bbox":bbox})

        self.dataset = dataset

        """
        self.dataset = [{"file_name": os.path.join("./refcocog/images/", f'{"_".join(elem["file_name"].split("_")[:3])}.jpg'),
                            "caption": tokenizer(
                                    elem["sentences"][0]["raw"], padding = "max_length", truncation=True, max_length=70,
                                    return_attention_mask = False
                                ),
                            "bbox": annotations[int(elem["file_name"].split("_")[3][:-4])]}
                        for elem in [d for d in refs if d["split"]==split]]
        """
    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

    def __call__(self, idx):
        print(json.dumps(self.dataset[idx], indent=4))


In [7]:
with open("./refcocog/annotations/refs(umd).p", "rb") as fp:
  refs = pickle.load(fp)

# 'annotations' will be a dict object mapping the 'annotation_id' to the 'bbox' to make search faster
with open("./refcocog/annotations/instances.json", "rb") as fp:
  data = json.load(fp)
  annotations = dict(sorted({ann["id"]: ann["bbox"] for ann in data["annotations"]}.items()))

In [9]:
def pad_image(image):
    """
    Performs bottom-right padding of the original image to 640x640 (max size of images in the dataset).
    Bottom-right padding prevents corruption of bounding boxes.

    ### Arguments
    image: a PIL.Image to transform
    """
    padded_width, padded_height = 640, 640

    padded_image = Image.new(image.mode, (padded_width, padded_height), (0, 0, 0))
    padded_image.paste(image, (0, 0))

    return padded_image

def collate_fn(batch):
    images = []
    #Stores all images in a list
    for sample in batch:
        image = Image.open(sample["file_name"]).convert("RGB")
        image = pad_image(image=image)
        images.append(transform(image))

    images = torch.stack(images, dim=0)

    data = {}
    for key in batch[0].keys():
        #if key != "file_name":
        #    data[key] = [sample[key] for sample in batch]
        data[key] = [sample[key] for sample in batch]
    return images, data

transform = transforms.Compose([
    transforms.ToTensor(),
])

# create dataset and dataloader
dataset = RefCOCOG(refs, annotations, split="test")
print(dataset[0])
print(dataset[1])
print(dataset[2])
print(dataset[3])
print("---------------------------------------------------")
#plt.imshow(Image.open(dataset[2]["file_name"]))
dataloader = DataLoader(dataset, batch_size=1, collate_fn=collate_fn)

{'file_name': './refcocog/images/COCO_train2014_000000380440.jpg', 'raw': 'the man in yellow coat', 'bbox': [374.31, 65.06, 136.04, 201.94]}
{'file_name': './refcocog/images/COCO_train2014_000000380440.jpg', 'raw': 'Skiier in red pants.', 'bbox': [374.31, 65.06, 136.04, 201.94]}
{'file_name': './refcocog/images/COCO_train2014_000000419645.jpg', 'raw': 'There is red colored truck in between the other trucks', 'bbox': [93.95, 83.29, 504.61, 290.57]}
{'file_name': './refcocog/images/COCO_train2014_000000419645.jpg', 'raw': 'A shiny red vintage pickup truck', 'bbox': [93.95, 83.29, 504.61, 290.57]}
---------------------------------------------------


# Model

In [10]:
yolo_model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
yolo_model.to(device)

Using cache found in /Users/mattiacarolo/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2024-6-2 Python-3.9.6 torch-2.3.0 CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


AutoShape(
  (model): DetectMultiBackend(
    (model): DetectionModel(
      (model): Sequential(
        (0): Conv(
          (conv): Conv2d(3, 32, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2))
          (act): SiLU(inplace=True)
        )
        (1): Conv(
          (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (act): SiLU(inplace=True)
        )
        (2): C3(
          (cv1): Conv(
            (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (cv2): Conv(
            (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (cv3): Conv(
            (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (m): Sequential(
            (0): Bottleneck(
              (cv1): Conv(
                (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
  